# Explore TPC-H source tables

In [0]:
# Explore available tables in the TPC-H schema
tables = spark.sql("show tables in samples.tpch")
display(tables)

In [0]:
# Compare row counts between samples.tpch and tpch_sf1 to identify scale factor
tables = ['customer', 'orders', 'lineitem', 'part',
          'partsupp', 'supplier', 'nation', 'region']

for t in tables:
    a = spark.table(f"samples.tpch.{t}").count()
    b = spark.table(f"samples.tpch_sf1.{t}").count()
    print(f"{t:10} tpch={a:>12,}  tpch_sf1={b:>12,}")

_Dataset scale factors:_

`samples.tpch` uses scale factor 5 (SF5), while `samples.tpch_sf1` uses scale factor 1 (SF1). The row-count comparison above shows about 5x more rows in the larger tables.

The existing `samples.tpch` validation results are kept as they are.
For framework development, `samples.tpch_sf1` will be used.
Later, both scale factors will be used to compare validation run times.

In [0]:
# Inspect customer table schema and sample records
df = spark.table("samples.tpch.customer")
df.printSchema()
display(df.limit(10))

In [0]:
# Check primary-key uniqueness and nulls in customer
from pyspark.sql import functions as F

df = spark.table("samples.tpch.customer")

df.agg(
    F.count("*").alias("row_count"),
    F.countDistinct("c_custkey").alias("distinct_key_count"),
    F.sum(
        F.when(F.col("c_custkey").isNull(), 1).otherwise(0) 
    ).alias("null_key_count")
).show()

In [0]:
# Check composite-key uniqueness and nulls in partsupp
df = spark.table("samples.tpch.partsupp")
df.agg(
    F.count("*").alias("row_count"),
    F.countDistinct("ps_partkey", "ps_suppkey").alias("distinct_key_count"),
    F.sum(
        F.when(
            F.col("ps_partkey").isNull() |
            F.col("ps_suppkey").isNull(),
            1
        ).otherwise(0)
    ).alias("null_key_count")
).show()

In [0]:
# Define reusable primary-key validation function (metrics only)
from functools import reduce
from operator import or_
from pyspark.sql import functions as F

def validate_primary_key(table_name, key_columns):
    df = spark.table(table_name)

    null_condition = reduce(
        or_,
        [F.col(c).isNull() for c in key_columns]
    )

    metrics = df.agg(
        F.count("*").alias("row_count"),
        F.countDistinct(*key_columns).alias("distinct_key_count"),
        F.sum(
            F.when(null_condition, 1).otherwise(0)
        ).alias("null_key_count")
    ).first()

    return metrics

In [0]:
# Define reusable primary-key validation function with PASS/FAIL classification
from pyspark.sql import functions as F

def validate_primary_key(table_name, key_columns):
    df = spark.table(table_name)

    null_condition = F.lit(False)

    for column_name in key_columns:
        null_condition = (
            null_condition | F.col(column_name).isNull()
        )

    metrics = df.agg(
        F.count("*").alias("row_count"),
        F.countDistinct(*key_columns).alias("distinct_key_count"),
        F.sum(
            F.when(null_condition, 1).otherwise(0)
        ).alias("null_key_count")
    ).first()

    row_count = metrics["row_count"]
    distinct_key_count = metrics["distinct_key_count"]
    null_key_count = metrics["null_key_count"]

    duplicate_count = row_count - null_key_count - distinct_key_count

    if null_key_count == 0 and duplicate_count == 0:
        test_result = "PASS"
    else:
        test_result = "FAIL"

    return {
        "table_name": table_name,
        "key_columns": key_columns,
        "row_count": row_count,
        "distinct_key_count": distinct_key_count,
        "null_key_count": null_key_count,
        "duplicate_count": duplicate_count,
        "test_result": test_result
    }

In [0]:
# Test reusable validation function
customer_result = validate_primary_key(
    "samples.tpch.customer",
    ["c_custkey"]
)

partsupp_result = validate_primary_key(
    "samples.tpch.partsupp",
    ["ps_partkey", "ps_suppkey"]
)

print(customer_result)
print(partsupp_result)